# Случайная карта + адресный поиск

Проверка гипотезы на двух сохранённых 600-текстовых пулах. Здесь реальные JSD уже измерены раньше, но GP и контрольные модели обучаются заново. Оба пула development. GPU не требуется.

Для каждого seed: SRS192 → карта θ; те же метки → общий предиктор → следующие60 кандидатов. Контроль добирает60 случайно. Сравниваем поиск и карту при общем бюджете252, а также GP с KRR, Bayesian Ridge и простой кластерной оценкой.

In [ ]:
%pip -q install numpy==2.1.3 scipy==1.16.3 scikit-learn==1.6.1 threadpoolctl==3.7.0 matplotlib==3.11.2


## Загрузить данные

Выбери `qwen_srs_gp_search_inputs.zip`. Архив содержит сохранённые признаки, JSD и планы, а не готовые результаты нового опыта.

In [ ]:
from pathlib import Path
import zipfile
from google.colab import files

WORK = Path("/content/gp_srs_search")
WORK.mkdir(exist_ok=True)
uploaded = files.upload()
input_zip = next(iter(uploaded))
with zipfile.ZipFile(input_zip) as archive:
    archive.extractall(WORK)


In [ ]:
from pathlib import Path
import argparse
import hashlib
import json
import platform
import sys
import time
import warnings
import numpy as np
import scipy
import sklearn
from scipy.stats import spearmanr
from sklearn.cluster import KMeans
from sklearn.compose import TransformedTargetRegressor
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import ConstantKernel, Matern, WhiteKernel
from sklearn.kernel_ridge import KernelRidge
from sklearn.linear_model import BayesianRidge
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from threadpoolctl import threadpool_limits

HERE = WORK
PREDICTORS = ['gp', 'krr', 'bayesian_ridge', 'cluster_mean', 'domain_mean']


## Файлы и порядок кандидатов

In [ ]:
def read_json(path):
    return json.loads(Path(path).read_text(encoding='utf-8'))


def write_json(path, value):
    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2,
                                   allow_nan=False), encoding='utf-8')


def file_sha(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()


def top_indices(indices, scores, ids, count=10):
    return np.array(sorted(indices, key=lambda i: (-scores[i], str(ids[i])))[:count])



## Обучение только на initial192 и выбор кандидатов

In [ ]:
def fit_and_select(X, train_y, domains, ids, initial, remaining, seed):
    """Fit on initial labels only; no oracle/remaining labels in this interface."""
    X_train = X[initial]
    domain_means = {d: float(train_y[domains[initial] == d].mean())
                    for d in sorted(set(domains))}
    distances = np.sqrt(np.maximum(0, 2 - 2 * np.clip(X_train @ X_train.T, -1, 1)))
    median = float(np.median(distances[np.triu_indices(len(initial), 1)])) or 1.
    kernel = (ConstantKernel(1., (.01, 100.)) *
              Matern(median, (.05 * median, 10 * median), nu=1.5) +
              WhiteKernel(.05, (1e-5, 2.)))
    gp = GaussianProcessRegressor(kernel=kernel, normalize_y=True,
                                  random_state=seed, n_restarts_optimizer=0,
                                  alpha=1e-8, optimizer='fmin_l_bfgs_b')
    estimates = {}
    for kind in PREDICTORS:
        start = time.perf_counter()
        parameters = None
        cv_scores = None
        with warnings.catch_warnings(record=True) as caught:
            warnings.simplefilter('always')
            if kind == 'gp':
                model = gp.fit(X_train, train_y)
                parameters = str(model.kernel_)
            elif kind == 'krr':
                candidates = [
                    TransformedTargetRegressor(
                        regressor=KernelRidge(alpha=a, kernel='rbf', gamma=g),
                        transformer=StandardScaler())
                    for a in [.01, .1, 1.] for g in [.5, 2., 8.]
                ]
                cv = KFold(3, shuffle=True, random_state=seed)
                cv_scores = [float(cross_val_score(m, X_train, train_y,
                             scoring='neg_mean_absolute_error', cv=cv,
                             n_jobs=1).mean()) for m in candidates]
                model = candidates[int(np.argmax(cv_scores))].fit(X_train, train_y)
                parameters = str(model.regressor_.get_params())
            elif kind == 'bayesian_ridge':
                model = make_pipeline(StandardScaler(), BayesianRidge()).fit(X_train, train_y)
            elif kind == 'cluster_mean':
                # Fixed four BGE clusters within each domain. Labels only set
                # the cluster means; empty observed clusters use domain mean.
                cluster_labels = np.full(len(X), -1, dtype=int)
                cluster_means = {}
                for number, d in enumerate(sorted(domain_means)):
                    ix = np.flatnonzero(domains == d)
                    labels = KMeans(n_clusters=4, n_init=10,
                                    random_state=seed).fit_predict(X[ix])
                    cluster_labels[ix] = number * 4 + labels
                    for c in range(4):
                        key = number * 4 + c
                        observed = cluster_labels[initial] == key
                        # Restrict to this domain: other labels may be unfilled.
                        observed &= domains[initial] == d
                        cluster_means[key] = (float(train_y[observed].mean())
                                              if observed.any() else domain_means[d])
            fit_seconds = time.perf_counter() - start
            start_prediction = time.perf_counter()
            if kind in ['gp', 'krr', 'bayesian_ridge']:
                mu = model.predict(X)
            elif kind == 'cluster_mean':
                mu = np.array([cluster_means[c] for c in cluster_labels])
            else:
                mu = np.array([domain_means[d] for d in domains])
            prediction_seconds = time.perf_counter() - start_prediction
            start_selection = time.perf_counter()
            chosen = np.concatenate([
                top_indices(remaining[domains[remaining] == d], mu, ids)
                for d in sorted(domain_means)])
            selection_seconds = time.perf_counter() - start_selection
        estimates[kind] = dict(mu=mu, chosen=chosen,
                               fit_seconds=fit_seconds,
                               prediction_seconds=prediction_seconds,
                               selection_seconds=selection_seconds,
                               parameters=parameters, cv_scores=cv_scores,
                               warnings=[str(w.message) for w in caught])
    return estimates



## Оценка после сохранения выбора

In [ ]:
def theta_estimate(y, domains, indices, epsilon):
    return {d: float(np.mean(y[indices[domains[indices] == d]] <= epsilon))
            for d in sorted(set(domains))}


def theta_error(theta, truth):
    return 100 * float(np.mean([abs(theta[d] - truth[d]) for d in truth]))


def score_choices(y, domains, ids, token_counts, initial, remaining, chosen, mu=None):
    """Oracle is used only here, after predictions and choices are saved."""
    by_domain = {}
    all_labeled = np.r_[initial, chosen]
    for d in sorted(set(domains)):
        full = np.flatnonzero(domains == d)
        rest = remaining[domains[remaining] == d]
        acquired = chosen[domains[chosen] == d]
        true_rest_top = set(top_indices(rest, y, ids))
        true_full_top = set(top_indices(full, y, ids))
        by_domain[d] = dict(
            remaining_top10_recall=len(set(acquired) & true_rest_top) / 10,
            full_top10_recall_after_252=len(set(all_labeled) & true_full_top) / 10,
            mean_new_jsd=float(y[acquired].mean()),
            new_token_positions=int(token_counts[acquired].sum()))
    metrics = dict(
        next_top10_recall=float(np.mean([v['remaining_top10_recall'] for v in by_domain.values()])),
        full_top10_recall_after_252=float(np.mean([v['full_top10_recall_after_252'] for v in by_domain.values()])),
        mean_new_jsd=float(y[chosen].mean()),
        new_token_positions=int(token_counts[chosen].sum()),
        total_token_positions=int(token_counts[all_labeled].sum()),
        by_domain=by_domain)
    if mu is not None:
        error = y[remaining] - mu[remaining]
        metrics['unselected_mae'] = float(np.abs(error).mean())
        metrics['unselected_r2'] = float(1 - np.sum(error ** 2) /
                                         np.sum((y[remaining] - y[remaining].mean()) ** 2))
        for d in by_domain:
            rest = remaining[domains[remaining] == d]
            e = y[rest] - mu[rest]
            by_domain[d]['unselected_mae'] = float(np.abs(e).mean())
            by_domain[d]['unselected_r2'] = float(1 - np.sum(e ** 2) /
                                                 np.sum((y[rest] - y[rest].mean()) ** 2))
    return metrics



## Условные интервалы и сводка

In [ ]:
def paired_interval(values, seed):
    values = np.array(values, dtype=float)
    rng = np.random.default_rng(seed)
    means = values[rng.integers(0, len(values), size=(10000, len(values)))].mean(axis=1)
    return dict(mean=float(values.mean()),
                conditional_95=np.quantile(means, [.025, .975]).tolist(),
                positive_seeds=int((values > 0).sum()), n=len(values))


def aggregate_pool(records, protocol):
    methods = PREDICTORS + ['random']
    averaged = {}
    for method in methods:
        numeric = [k for k, v in records[0]['methods'][method].items()
                   if isinstance(v, (float, int))]
        averaged[method] = {k: float(np.mean([r['methods'][method][k] for r in records]))
                            for k in numeric}
        averaged[method]['by_domain'] = {
            d: {k: float(np.mean([r['methods'][method]['by_domain'][d][k] for r in records]))
                for k in records[0]['methods'][method]['by_domain'][d]}
            for d in records[0]['methods'][method]['by_domain']}
    random_expected = 10 / 68
    comparisons = {}
    for number, method in enumerate(PREDICTORS):
        comparisons[method + '_minus_random_expectation'] = paired_interval(
            [r['methods'][method]['next_top10_recall'] - random_expected for r in records],
            protocol['bootstrap_seed'] + number)
        comparisons[method + '_minus_random_draw'] = paired_interval(
            [r['methods'][method]['next_top10_recall'] - r['methods']['random']['next_top10_recall']
             for r in records], protocol['bootstrap_seed'] + 10 + number)
    for number, control in enumerate(['krr', 'bayesian_ridge', 'cluster_mean']):
        comparisons['gp_minus_' + control] = paired_interval(
            [r['methods']['gp']['next_top10_recall'] - r['methods'][control]['next_top10_recall']
             for r in records], protocol['bootstrap_seed'] + 20 + number)
    comparisons['random252_theta_error_minus_hybrid192_pp'] = paired_interval(
        [r['theta_random252_error_pp'] - r['theta_initial192_error_pp'] for r in records],
        protocol['bootstrap_seed'] + 30)
    primary = comparisons['gp_minus_random_expectation']
    if primary['conditional_95'][0] > 0:
        decision = 'positive_search_signal_on_this_opened_pool'
    elif primary['conditional_95'][1] < 0:
        decision = 'gp_search_worse_than_random_expectation_on_this_opened_pool'
    else:
        decision = 'inconclusive_search_signal_on_this_opened_pool'
    return dict(decision=decision, methods=averaged, comparisons=comparisons,
                exact_random_expected_recall=random_expected,
                theta_initial192_error_pp=float(np.mean([r['theta_initial192_error_pp'] for r in records])),
                theta_random252_error_pp=float(np.mean([r['theta_random252_error_pp'] for r in records])))



## Запуск всех 40 планов

In [ ]:
def run_experiment(root=HERE):
    start = time.perf_counter()
    protocol = read_json(root / 'protocol.json')
    output = root / 'results'
    output.mkdir(exist_ok=True)
    provenance = dict(protocol_sha256=file_sha(root / 'protocol.json'),
                      script_sha256=file_sha(root / 'experiment.py'),
                      python=platform.python_version(), numpy=np.__version__,
                      scipy=scipy.__version__, sklearn=sklearn.__version__,
                      new_qwen_measurements=0, threads=1)
    results = {}
    with threadpool_limits(limits=1):
        # Warm up libraries without access to pool labels.
        dummy = np.random.default_rng(0).normal(size=(40, 8))
        BayesianRidge().fit(dummy, dummy[:, 0]).predict(dummy)
        for pool_name, pool_info in protocol['pools'].items():
            folder = root / 'inputs' / pool_name
            for filename, digest in pool_info['sha256'].items():
                assert file_sha(folder / filename) == digest, filename
            with np.load(folder / 'features.npz') as a:
                X, ids = a['X'], a['ids']
            with np.load(folder / 'oracle.npz') as a:
                y, domains = a['y'], a['domains']
                assert np.array_equal(ids, a['ids'])
            counts = np.load(folder / 'token_counts.npy')
            plans = read_json(folder / 'plans.json')
            truth = theta_estimate(y, domains, np.arange(len(y)), protocol['epsilon'])
            pool_output = output / pool_name
            pool_output.mkdir(exist_ok=True)
            records = []
            for seed in protocol['seeds']:
                plan = plans[str(seed)]
                initial = np.array(plan['initial'], dtype=int)
                remaining = np.setdiff1d(np.arange(len(X)), initial)
                random_chosen = np.array(plan['random_next'], dtype=int)
                assert len(initial) == 192 and len(remaining) == 408
                predictions = fit_and_select(X, y[initial].copy(), domains, ids,
                                             initial, remaining, seed)
                # Save selections before calling the oracle evaluator.
                arrays = dict(initial=initial, remaining=remaining,
                              random_chosen=random_chosen, train_y=y[initial])
                for method, v in predictions.items():
                    arrays[method + '_mu'] = v['mu']
                    arrays[method + '_chosen'] = v['chosen']
                candidate_file = pool_output / f'seed_{seed}_choices.npz'
                np.savez_compressed(candidate_file, **arrays)
                initial_theta = theta_estimate(y, domains, initial, protocol['epsilon'])
                random_theta = theta_estimate(y, domains, np.r_[initial, random_chosen],
                                              protocol['epsilon'])
                record = dict(seed=seed, choices_sha256=file_sha(candidate_file),
                              train_label_sha256=hashlib.sha256(y[initial].tobytes()).hexdigest(),
                              theta_initial192=initial_theta, theta_random252=random_theta,
                              theta_initial192_error_pp=theta_error(initial_theta, truth),
                              theta_random252_error_pp=theta_error(random_theta, truth),
                              initial_token_positions=int(counts[initial].sum()), methods={})
                for method, v in predictions.items():
                    metrics = score_choices(y, domains, ids, counts, initial, remaining,
                                            v['chosen'], v['mu'])
                    record['methods'][method] = {**metrics, **{k: v[k] for k in
                        ['fit_seconds', 'prediction_seconds', 'selection_seconds',
                         'parameters', 'cv_scores', 'warnings']}}
                record['methods']['random'] = score_choices(y, domains, ids, counts,
                                                            initial, remaining, random_chosen)
                records.append(record)
                write_json(pool_output / 'seed_results.json', records)
                print(f'{pool_name} seed={seed} next_top10 '
                      f'GP={record["methods"]["gp"]["next_top10_recall"]:.3f} '
                      f'KRR={record["methods"]["krr"]["next_top10_recall"]:.3f} '
                      f'random={record["methods"]["random"]["next_top10_recall"]:.3f}', flush=True)
            summary = aggregate_pool(records, protocol)
            summary['truth_theta'] = truth
            write_json(pool_output / 'summary.json', summary)
            results[pool_name] = summary
    report = dict(status='completed', scope='offline development on two previously opened pools',
                  provenance=provenance, pools=results, seconds=time.perf_counter() - start,
                  limitations=protocol['limitations'])
    write_json(output / 'summary.json', report)
    print('COMPLETED', json.dumps({
        'seconds': report['seconds'], 'new_qwen_measurements': 0,
        'decisions': {k: v['decision'] for k, v in results.items()}}, ensure_ascii=False), flush=True)
    return report



In [ ]:
report = run_experiment(WORK)


## Результат

Recall — доля найденной истинной десятки среди оставшихся68 текстов домена. θ-error — ошибка карты в процентных пунктах. Все интервалы условны на открытый пул; вторичные сравнения описательные.

In [ ]:
for pool, result in report["pools"].items():
    print("\n", pool, result["decision"])
    for method, metrics in result["methods"].items():
        print(f"{method:16s} next recall={100*metrics["next_top10_recall"]:.2f}%")
    print("theta-error hybrid192:", result["theta_initial192_error_pp"])
    print("theta-error random252:", result["theta_random252_error_pp"])
    print("GP minus KRR:", result["comparisons"]["gp_minus_krr"])


In [ ]:
import shutil
result_zip = shutil.make_archive("/content/qwen_srs_gp_search_results", "zip", WORK / "results")
files.download(result_zip)
